In [1]:
import subprocess
import pandas
import io

pandas.options.mode.copy_on_write = True

result = subprocess.run(["cloc", "--include-lang=Scala", "--by-file", "--csv", "../"], capture_output=True)
cloc = pandas.read_csv(io.BytesIO(result.stdout))
cloc = cloc[["filename", "code"]]
cloc.drop(cloc.tail(1).index, inplace=True)

cloc

,filename,code
0,../sturdy-core/src/main/scala/sturdy/values/in...,990
1,../sturdy-wasm/src/main/scala/sturdy/language/...,641
2,../sturdy-core/src/main/scala/sturdy/values/in...,561
3,../sturdy-core/src/test/scala/sturdy/values/in...,516
4,../sturdy-core/src/test/scala/sturdy/control/T...,421
...,...,...
375,../sturdy-tip/src/test/scala/sturdy/language/t...,0
376,../sturdy-tip/src/test/scala/sturdy/language/t...,0
377,../sturdy-tip/src/test/scala/sturdy/language/t...,0
378,../sturdy-wasm-benchmarks/src/main/scala/sturd...,0


In [117]:
def sturdy_wasm(re: str):
    return cloc.filename.str.contains('sturdy-wasm/src/main/scala/sturdy/language/wasm/'+re, regex= True, na=False)

def sturdy_tip(re: str):
    return cloc.filename.str.contains('sturdy-tip/src/main/scala/sturdy/language/tip/'+re, regex= True, na=False)

def sturdy_apron(re: str):
    return cloc.filename.str.contains('sturdy-apron/src/main/scala/sturdy/'+re, regex= True, na=False)

def sturdy_core(re: str):
    return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)
    
def export_data(res):
    dict = {
        'sumCode':  res['code'].sum(),
        'sumReuse': res[res['reuse'] == True]['code'].sum(),
        'sumNoReuse': res[res['reuse'] == False]['code'].sum(),
        'sumLanguageIndependent': res[res['language-independent'] == True]['code'].sum(),
        'sumLanguageSpecific': res[res['language-independent'] == False]['code'].sum(),
        'sumLanguageIndependentReuse': res[(res['language-independent'] == True) & (res['reuse'] == True)]['code'].sum(),
        'sumLanguageIndependentNoReuse': res[(res['language-independent'] == True) & (res['reuse'] == False)]['code'].sum(),
        'sumLanguageSpecificReuse': res[(res['language-independent'] == False) & (res['reuse'] == True)]['code'].sum(),
        'sumLanguageSpecificNoReuse': res[(res['language-independent'] == False) & (res['reuse'] == False)]['code'].sum(),
        'sumReuseOrLanguageIndependent': res[(res['language-independent'] == True) | (res['reuse'] == True)]['code'].sum()
    }
    
    new_dict = {}
    for key, val in dict.items():
        new_dict[key] = str(dict[key])+' loc'
        
        new_key = 'percent'+key.removeprefix('sum')
        new_dict[new_key] = "{:.1f}\\%".format(dict[key] / dict['sumCode'] * 100)
    dict = new_dict
    
    for key, val in dict.items():
        print('\\def\\'+key+'{'+val+'\\xspace}')

def print_filenames(df):
    for filename in df.sort_values(by=['filename'])['filename']:
        print(filename)

In [93]:
# Basic functionality all analyses need.
basic_functionality = cloc[
    sturdy_core('data/.*') |
    sturdy_core('effect/Effect.*') |
    sturdy_core('effect/ComputationJoiner.scala') |
    sturdy_core('effect/TrySturdy.scala') |
    sturdy_core('fix/Combinators.scala') |
    sturdy_core('fix/Contextual.scala') |
    sturdy_core('fix/Fixpoint.scala') |
    sturdy_core('fix/Stack.scala') |
    sturdy_core('fix/StackedStates.scala') |
    sturdy_core('fix/State.scala') |
    sturdy_core('fix/context/Sensitivity.scala') |
    sturdy_core('fix/iter/.*') |
    sturdy_core('values/Join.scala') |
    sturdy_core('values/Topped.scala') |
    sturdy_core('values/references/AbstractAddr.scala')
]
basic_functionality["language-independent"] = True
basic_functionality["reuse"] = True

In [122]:
# Tip Analysis

# Language-specific, reuse
tip_lang_specific_reuse = cloc[
    # We don't count code of parser and AST
    sturdy_tip('GenericInterpreter') |
    sturdy_tip('Interpreter.scala') |
    sturdy_tip('abstractions/Functions.scala') |
    sturdy_tip('abstractions/Records.scala') |
    sturdy_tip('abstractions/Control.scala') |
    sturdy_tip('abstractions/Fix.scala')
]
tip_lang_specific_reuse["language-independent"] = False
tip_lang_specific_reuse["reuse"] = True

# Language-specific, no reuse
tip_lang_specific_no_reuse = cloc[
    sturdy_tip('analysis/RelationalAnalysis.scala')
]
tip_lang_specific_no_reuse["language-independent"] = False
tip_lang_specific_no_reuse["reuse"] = False

# Language-independent, reuse
tip_lang_independent_reuse = pandas.concat([basic_functionality, cloc[
    sturdy_core('effect/allocation/AAllocatorFromContext.scala') |
    sturdy_core('effect/allocation/Allocator.scala') |
    sturdy_core('effect/callframe/(Decidable)?CallFrame.scala') |
    sturdy_core('effect/store/Store.scala') |
    sturdy_core('effect/store/AStoreThreaded.scala') |
    sturdy_core('effect/print/Print(Bound)?.scala') |
    sturdy_core('effect/failure/(Collected|A)?Failure(s)?.scala') |
    sturdy_core('effect/failure/AFallible.scala') |
    sturdy_core('effect/failure/Assert.scala') |
    sturdy_core('effect/userinput/(A)?UserInput.scala') |
    sturdy_core('values/booleans/(Lifted|Topped)?Boolean(Branching|Ops).scala') |
    sturdy_core('values/integer/(Lifted)?IntegerOps.scala') |
    sturdy_core('values/functions/(Lifted|Powerset)?FunctionOps.scala') |
    sturdy_core('values/ordering/(Lifted)?(Unsigned)?(Ordering|Eq)Ops.scala') |
    sturdy_core('values/references/AbstractReference.scala') |
    sturdy_core('values/references/(Lifted)?ReferenceOps.scala') |
    sturdy_core('values/records/ARecords.scala') |
    sturdy_core('values/records/(Lifted)?RecordOps.scala') |
    sturdy_core('values/types/BaseType.scala') |
    sturdy_core('values/Powerset.scala')
]], axis=0)
tip_lang_independent_reuse["language-independent"] = True
tip_lang_independent_reuse["reuse"] = True

# Language-independent, no reuse
tip_lang_independent_no_reuse = cloc[
    sturdy_core('values/references/RecencyAddr.scala') |
    sturdy_core('effect/store/RecencyStore.scala') |
    sturdy_apron('apron/.*') |
    sturdy_apron('effect/callframe/.*') |
    sturdy_apron('effect/store/.*') |
    sturdy_apron('values/(booleans|integer|ordering)/.*') 
]
tip_lang_independent_no_reuse["language-independent"] = True
tip_lang_independent_no_reuse["reuse"] = False

tip = pandas.concat([tip_lang_specific_reuse, tip_lang_specific_no_reuse, tip_lang_independent_reuse, tip_lang_independent_no_reuse])

print("%%% Tip Analysis %%%")
export_data(tip)

%%% Tip Analysis %%%
\def\sumCode{5790 loc\xspace}
\def\percentCode{100.0\%\xspace}
\def\sumReuse{2950 loc\xspace}
\def\percentReuse{50.9\%\xspace}
\def\sumNoReuse{2840 loc\xspace}
\def\percentNoReuse{49.1\%\xspace}
\def\sumLanguageIndependent{5101 loc\xspace}
\def\percentLanguageIndependent{88.1\%\xspace}
\def\sumLanguageSpecific{689 loc\xspace}
\def\percentLanguageSpecific{11.9\%\xspace}
\def\sumLanguageIndependentReuse{2500 loc\xspace}
\def\percentLanguageIndependentReuse{43.2\%\xspace}
\def\sumLanguageIndependentNoReuse{2601 loc\xspace}
\def\percentLanguageIndependentNoReuse{44.9\%\xspace}
\def\sumLanguageSpecificReuse{450 loc\xspace}
\def\percentLanguageSpecificReuse{7.8\%\xspace}
\def\sumLanguageSpecificNoReuse{239 loc\xspace}
\def\percentLanguageSpecificNoReuse{4.1\%\xspace}
\def\sumReuseOrLanguageIndependent{5551 loc\xspace}
\def\percentReuseOrLanguageIndependent{95.9\%\xspace}


/tmp/ipykernel_6759/2585529434.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)
/tmp/ipykernel_6759/2585529434.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-apron/src/main/scala/sturdy/'+re, regex= True, na=False)


In [123]:
# WebAssembly Analysis

# Language-specific, reuse
wasm_lang_specific_reuse = cloc[
    # We don't count code of parser and AST
    sturdy_wasm('generic/') |
    sturdy_wasm('Interpreter.scala') |
    sturdy_wasm('Properties.scala') |
    sturdy_wasm('abstractions/ExceptionByTarget.scala') |
    sturdy_wasm('abstractions/ControlFlow.scala')
]
wasm_lang_specific_reuse["language-independent"] = False
wasm_lang_specific_reuse["reuse"] = True


# Language-specific, no reuse
wasm_lang_specific_no_reuse = cloc[
    sturdy_wasm('analyses/RelationalAnalysis.scala') |
    sturdy_wasm('abstractions/Relational.*')
]
wasm_lang_specific_no_reuse["language-independent"] = False
wasm_lang_specific_no_reuse["reuse"] = False

# Language-independent, reuse
wasm_lang_independent_reuse = pandas.concat([basic_functionality, tip_lang_independent_no_reuse, cloc[
    sturdy_core('effect/allocation/AAllocatorFromContext.scala') |
    sturdy_core('effect/allocation/Allocator.scala') |
    sturdy_core('effect/callframe/DecidableCallFrame.scala') |
    sturdy_core('effect/bytememory/Memory.scala') |
    sturdy_core('effect/operandstack/(Decidable)?OperandStack.scala') |
    sturdy_core('effect/symboltable/(Decidable|Interval)?SymbolTable.scala') |
    sturdy_core('effect/except/JoinedExcept.scala') |
    sturdy_core('effect/except/Except.scala') |
    sturdy_core('effect/failure/CollectedFailures.scala') |
    sturdy_core('effect/failure/AFallible.scala') |
    sturdy_core('effect/failure/Failure.scala') |
    sturdy_core('values/booleans/(Lifted)?Boolean(Branching|Ops).scala') |
    sturdy_core('values/config/.*') |
    sturdy_core('values/convert/(Lifted|Transitive)?Convert.scala') |
    sturdy_core('values/exceptional/Exceptional(ByTarget)?.scala') |
    sturdy_core('values/floating/(Lifted)?FloatOps.scala') |
    sturdy_core('values/integer/(Lifted)?IntegerOps.scala') |
    sturdy_core('values/ordering/(Lifted)?(Unsigned)?(Ordering|Eq)Ops.scala') |
    sturdy_core('values/types/BaseType.scala')
]], axis=0)
wasm_lang_independent_reuse["language-independent"] = True
wasm_lang_independent_reuse["reuse"] = True


# Language-independent, no reuse
wasm_lang_independent_no_reuse = cloc[
    sturdy_apron('effect/bytememory/.*') |
    sturdy_apron('effect/callframe/.*') |
    sturdy_apron('effect/stack/.*') |
    sturdy_apron('effect/symboltable/.*') |
    sturdy_apron('values/convert/.*') |
    sturdy_apron('values/floating/.*')
]
wasm_lang_independent_no_reuse["language-independent"] = True
wasm_lang_independent_no_reuse["reuse"] = False

print_filenames(wasm_lang_independent_no_reuse)

wasm = pandas.concat([wasm_lang_specific_reuse, wasm_lang_specific_no_reuse, wasm_lang_independent_reuse, wasm_lang_independent_no_reuse])

print("%%% WebAssembly Analysis %%%")
export_data(wasm)


../sturdy-apron/src/main/scala/sturdy/effect/bytememory/RelationalMemory.scala
../sturdy-apron/src/main/scala/sturdy/effect/callframe/RelationalCallFrame.scala
../sturdy-apron/src/main/scala/sturdy/effect/stack/RelationalStack.scala
../sturdy-apron/src/main/scala/sturdy/effect/symboltable/RelationalSymbolTable.scala
../sturdy-apron/src/main/scala/sturdy/values/convert/RelationalConvert.scala
../sturdy-apron/src/main/scala/sturdy/values/floating/RelationalFloatOps.scala
%%% WebAssembly Analysis %%%
\def\sumCode{8454 loc\xspace}
\def\percentCode{100.0\%\xspace}
\def\sumReuse{6722 loc\xspace}
\def\percentReuse{79.5\%\xspace}
\def\sumNoReuse{1732 loc\xspace}
\def\percentNoReuse{20.5\%\xspace}
\def\sumLanguageIndependent{6151 loc\xspace}
\def\percentLanguageIndependent{72.8\%\xspace}
\def\sumLanguageSpecific{2303 loc\xspace}
\def\percentLanguageSpecific{27.2\%\xspace}
\def\sumLanguageIndependentReuse{5108 loc\xspace}
\def\percentLanguageIndependentReuse{60.4\%\xspace}
\def\sumLanguageIndepe

/tmp/ipykernel_6759/2585529434.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)
